In [1]:
import pandas as pd
from pathlib import Path

pwd = Path.cwd()

output_path = pwd / "Summaries-2"
output_path.mkdir(exist_ok=True)

results_path = pwd / "CollectedResults"

results = [
    (pd.read_csv(f), f.stem.title())
    for f in results_path.iterdir()
    if f.suffix != ".txt"
]

results[0][0]

,model_name,Normal_accuracy,Normal_f1_score,Normal_precision,Normal_recall,Normal_specificity,Normal_matthews,Normal_auc,SMOTE_accuracy,SMOTE_f1_score,...,Smote_hell,Ada_hell,Gan_hell,SG_hell,AG_hell,Smote_time(ms),Ada_time,Gan_time,SG_time,AG_time
0,logistic_regression_classifier,0.966332,0.966283,0.966272,0.966328,0.956765,0.925816,0.961546,0.969261,0.969245,...,0.0,0.0,0.0,0.0,0.0,5.615421,9.350124,6960.732894,8453.620447,8156.135413
1,logistic_regression_classifier,0.966327,0.966283,0.966274,0.966332,0.956763,0.925818,0.961547,0.970719,0.970724,...,0.0,0.0,0.0,0.0,0.0,5.615423,9.350117,6960.732892,8453.620451,8156.135409
2,logistic_regression_classifier,0.966329,0.966281,0.966274,0.966327,0.956763,0.925815,0.961545,0.963398,0.963351,...,0.0,0.0,0.0,0.0,0.0,5.615423,9.350121,6960.732898,8453.620451,8156.135409
3,logistic_regression_classifier,0.966334,0.966278,0.966273,0.966329,0.956764,0.925818,0.961545,0.966331,0.966314,...,0.0,0.0,0.0,0.0,0.0,5.615418,9.350120,6960.732892,8453.620451,8156.135415
4,logistic_regression_classifier,0.966333,0.966283,0.966269,0.966332,0.956760,0.925818,0.961548,0.969258,0.969241,...,0.0,0.0,0.0,0.0,0.0,5.615418,9.350117,6960.732892,8453.620447,8156.135413
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,simple_vector_classifier,0.963398,0.963455,0.963546,0.963400,0.959049,0.919825,0.961225,0.969259,0.969388,...,0.0,0.0,0.0,0.0,0.0,4.948005,8.793408,6839.340625,8155.096323,8145.874159
86,simple_vector_classifier,0.963404,0.963450,0.963543,0.963401,0.959048,0.919829,0.961230,0.964863,0.964962,...,0.0,0.0,0.0,0.0,0.0,4.948002,8.793409,6839.340629,8155.096318,8145.874155
87,simple_vector_classifier,0.963401,0.963455,0.963548,0.963400,0.959050,0.919825,0.961225,0.966328,0.966436,...,0.0,0.0,0.0,0.0,0.0,4.948006,8.793403,6839.340624,8155.096324,8145.874156
88,simple_vector_classifier,0.963405,0.963456,0.963549,0.963404,0.959053,0.919823,0.961226,0.969259,0.969388,...,0.0,0.0,0.0,0.0,0.0,4.948004,8.793406,6839.340626,8155.096319,8145.874156


In [2]:
def get_single_timming(df: pd.DataFrame):
    time_cols = df.filter(regex="time", axis="columns").columns

    df[time_cols] = df[time_cols] / 1000

    df = df.filter(regex="time|model_name", axis="columns")

    summary = (
        df.groupby("model_name")
        .agg(["sum"])
        .stack(
            level=1,future_stack=True
        )
        .reset_index()
    )

    summary.loc[summary["model_name"].duplicated(), "model_name"] = ""

    return summary


In [3]:
from typing import List, Tuple

def get_all_timmings(results: List[Tuple[pd.DataFrame,str]]):
    summary = pd.DataFrame()

    for result, name in results:
        single_summary = get_single_timming(result)
        single_summary.insert(0, "Dataset",name)

        summary = pd.concat([summary,single_summary])
        summary.loc[summary["Dataset"].duplicated(), "Dataset"] = ""
    
    return summary

In [4]:
get_all_timmings(results).rename(columns={"level_1":"stats"}).to_csv(output_path / "timmings2.csv")